STEP01: IMPORTING REQUIRED LIBRARIES FOR EDA

1. NumPy       :-  for numerical operations and mathematical calculations
2. Pandas      :-  for data loading, manipulation, and analysis
3. Matplotlib  :-  for basic data visualization (charts and graphs)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

STEP02: DATA LOADING

- Reading the CSV file and storing it in a DataFrame called 'dataset'.

In [ ]:
dataset = pd.read_csv("data_science_salaries.csv")
dataset

STEP03: DATASET INSPECTION

- In this step, I explore the dataset to understand its structure,
- data types, missing values, and basic patterns before analysis

In [ ]:
# 01 Check shape of dataset (rows, columns)
# This tells how big the dataset is
data_shape = dataset.shape
print(f"1. Dataset Shape (rows, columns): {data_shape}\n")


# 02 View column names
# Helps understand what features are available
data_columns = dataset.columns
print(f"2. Dataset Columns:")
for col in data_columns:
    print(col)

In [ ]:
# 03 View first 5 rows
# Gives a quick idea of how data looks
dataset.head()

In [ ]:
# 04 Get basic info about dataset
# Shows data types, non-null values, memory usage
dataset.info()

In [ ]:
# 05 Statistical summary of numeric columns
# Helps understand distribution (mean, min, max, etc.)
dataset.describe()

In [ ]:
# 06 Check missing values in each column
# Helps identify data quality issues
missing_values = dataset.isnull().sum()
print(f"1. Missing Values of Dataset(Column-wise):\n{missing_values}\n")


# 07 Check duplicate rows
# Ensures there is no repeated data
duplicate_count = dataset.duplicated().sum()
print(f"2. Duplicate Rows Count: {duplicate_count}")


In [ ]:
# 08 Unique values in important columns
# Helps understand category diversity

print(f"job_title: {dataset["job_title"].nunique()}")
print(f"experience_level: {dataset["experience_level"].nunique()}")
print(f"employment_type: {dataset["employment_type"].nunique()}")
print(f"work_models: {dataset["work_models"].nunique()}")
print(f"work_year: {dataset["work_year"].nunique()}")
print(f"company_size: {dataset["company_size"].nunique()}")


In [ ]:
# 09 Frequency of categorical columns
# Shows most common categories in each column with their counts sorted in descending order

print(f"{dataset["job_title"].value_counts()}\n")
print(f"{dataset["experience_level"].value_counts()}\n")
print(f"{dataset["employment_type"].value_counts()}\n")
print(f"{dataset["work_models"].value_counts()}\n")
print(f"{dataset["work_year"].value_counts()}\n")
print(f"{dataset["company_size"].value_counts()}\n")

STEP04: DATA CLEANING & MANIPULATION

- In this step, I prepare the dataset for analysis by cleaning and transforming it.
- I check for missing values, duplicate records, and any inconsistencies in the data.
- These steps help improve the overall quality and usability of the dataset

In [ ]:
# A. Created a copy of the original dataset to preserve raw data
# All cleaning and transformations will be done on this copy

dataset_copy = dataset.copy()
dataset_copy

In [ ]:
# B. Replaced abbreviations and inconsistent job titles with standardized full forms
# This ensures uniformity in job_title column for better analysis

dataset_copy["job_title"] = dataset_copy["job_title"].replace({
    "ML Engineer": "Machine Learning Engineer",
    "BI Developer": "Business Intelligence Developer",
    "BI Analyst": "Business Intelligence Analyst",
    "BI Data Analyst": "Business Intelligence Data Analyst",
    "Data Science": "Data Scientist",
    "Data Modeller": "Data Modeler",
    "ETL Developer": "ETL Engineer"
})

In [ ]:
# C. Created and Inserted "job_category" column
# This feature engineering step was performed to group similar job titles into meaningful categories
# It was necessary because the dataset contained a large number of unique job titles across similar roles,
# which made the data messy and difficult to analyze effectively during EDA and further analysis.



# step1: Defined func to assign category to each job_title respectively
def get_category(title):

    t = title.lower()

    
    # a. EXACT MATCH:
    exact_map = {
        "data engineer": "Data Engineering",
        "etl engineer": "Data Engineering",
        "big data engineer": "Data Engineering",
        "cloud data engineer": "Data Engineering",
        "azure data engineer": "Data Engineering",
        "software data engineer": "Data Engineering",
        "data developer": "Data Engineering",
        "data scientist": "Data Science",
        "data analyst": "Data Analysis",
        "machine learning engineer": "Machine Learning & AI",
        "ai engineer": "Machine Learning & AI",
        "business intelligence analyst": "Business Intelligence",
        "data architect": "Data Architecture",
        "big data architect": "Data Architecture",
        "data science manager": "Leadership & Management",
        "head of data science": "Leadership & Management",
        "director of data science": "Leadership & Management",
    }

    if t in exact_map:
        return exact_map[t]

    
    # b. KEYWORD MATCH:

    # Leadership 
    if any(k in t for k in ["lead", "head", "director", "manager", "principal", "staff"]):
        return "Leadership & Management"

    # Data Engineering
    if any(k in t for k in ["etl", "data engineer", "infrastructure", "integration", "pipeline", "devops"]):
        return "Data Engineering"

    # Data Science
    if any(k in t for k in ["scientist", "research scientist", "applied scientist"]):
        return "Data Science"

    # Machine Learning / AI
    if any(k in t for k in ["machine learning", "ml", "ai", "deep learning", "nlp", "computer vision"]):
        return "Machine Learning & AI"

    # Data Analysis
    if any(k in t for k in ["analyst", "analytics", "visualization", "insight"]):
        return "Data Analysis"

    # Business Intelligence
    if any(k in t for k in ["business intelligence", "bi", "power bi"]):
        return "Business Intelligence"

    # Data Architecture
    if any(k in t for k in ["architect", "modeler", "data architect"]):
        return "Data Architecture"

    
    # c. IF NONE, RETURN "Other"
    return "Other"





# step2: Applied categorization function to job_title column
new_col_values = dataset_copy["job_title"].apply(get_category)





# step3: Inserted job_category column at position 1 (right after job_title)
dataset_copy.insert(loc=1, column="job_category", value=new_col_values)
dataset_copy

In [ ]:
# D. Detecting and handling outliers in "salary_in_usd" for better analysis

# step1: check summary statistics to understand overall salary range and spot extreme values
dataset_copy["salary_in_usd"].describe()

In [ ]:
# step2: visualize distribution using a boxplot to understand spread and identify outliers
dataset_copy.boxplot(column="salary_in_usd")
plt.title("Salary Distribution")
plt.show()

# Observation:
# The salary distribution is right-skewed
# Most salaries are in the lower to mid range,
# while a few very high salaries are stretching the upper side.

# Important Note:
# Since the dataset contains different job levels,
# high salaries (especially for senior roles) may be valid and not true outliers.

# So instead of removing outliers from the whole dataset,
# I'll detect outliers within each job level separately.
# This ensures I don't remove valid high salaries by mistake

In [ ]:
# step3: Detect outliers using group-wise IQR (based on experience level)

def find_outliers_groupwise(df, group_col, target_col):
    outlier_df = pd.DataFrame()

    for group in df[group_col].unique():
        subset = df[df[group_col] == group]

        # Calculate IQR
        Q1 = subset[target_col].quantile(0.25)
        Q3 = subset[target_col].quantile(0.75)
        IQR = Q3 - Q1

        # Define bounds
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        print(f"{group}: {lower_bound}, {upper_bound}")

        # Identify outliers
        outliers = subset[
            (subset[target_col] < lower_bound) |
            (subset[target_col] > upper_bound)
        ]

        outlier_df = pd.concat([outlier_df, outliers])

    return outlier_df

print("1. Experience Level Lower & Upper Bounds:")
outliers_df = find_outliers_groupwise(dataset_copy, group_col="experience_level", target_col="salary_in_usd")
print(f"\n2. Total Outliers: {len(outliers_df)}\n")

# Observation:
# - Detected 133 outliers using group-wise IQR
# - Since salary is right-skewed, not all outliers are incorrect
# - High salaries (especially senior/executive) can be valid


# step4: Review extreme outliers
print(f"3. Top 20 Outliers:")
outliers_df.sort_values(by="salary_in_usd", ascending=False).head(20)

# Insight:
# - Found some mid-level salaries (~700k–750k) higher than executive level (~465k)
# - This breaks expected job hierarchy → likely data errors
# - These extreme values will be removed


In [ ]:
# step5: Filter out unrealistic extreme salaries and create clean dataset
clean_dataset = dataset_copy[dataset_copy["salary_in_usd"] < 600000]

# Final:
# - Removed only extreme unrealistic values (≥700k approx)
# - Kept valid high salaries (400k–465k range)
# - Dataset is now more realistic and balanced

# Dataset is ready for EDA / further analysis
clean_dataset

In [ ]:
# step6: Analyze clean dataset
print(clean_dataset.describe(), "\n")

# Insights:
# - After removing extreme outliers, salary distribution is more realistic
# - Maximum salary reduced from ~750k to ~465k
# - Mean and median are closer → skewness reduced

In [ ]:
clean_dataset.boxplot(column="salary_in_usd")
plt.title("Salary Distribution")
plt.show()

# Boxplot Insights:
# - Distribution remains right-skewed
# - Most salaries lie in a lower to mid range, with few high values
# - Remaining outliers represent valid high-paying roles, not errors
# - Dataset is now clean and ready for further analysis

STEP05: EXPLORATORY DATA ANALYSIS (EDA)

- In this step, I analyze the cleaned dataset to extract meaningful insights
- The goal is to understand patterns, trends, relationships, and visualize the data effectively.


KEY FOCUS AREAS:

- Identify most in-demand job roles
- Analyze salary distribution across different roles
- Compare salaries based on experience levels
- Understand work models (Remote / Hybrid / On-site)
- Study company size and hiring patterns
- Analyze relationship between year and salary (trend and correlation)

EDA SECTION 01: Analyzed Job Listings Across Different Data Domains

- This section analyzes the number of job listings across different data domains
- The goal is to understand which domains have the highest demand
- This helps us understand where most job opportunities are concentrated in the data field

In [ ]:
# Data
most_demand_job = clean_dataset["job_category"].value_counts()
x = most_demand_job.index
y = most_demand_job.values

# Global style
plt.rcParams['font.family'] = 'sans-serif'

# Figure
plt.figure(figsize=(12, 6), dpi=150)

# Plot
plt.bar(x, y, width=0.5, edgecolor="black", color="#52799C")

# Titles & Labels
plt.title("Job Listings Across Data Domains (Highest to Lowest)", 
            fontdict={"color":"#333333", "fontweight":"bold", "fontsize":15}, 
            pad=10)

plt.xlabel("Data Domain", 
            fontdict={"color":"#333333", "fontweight":"bold", "fontsize":13}, 
            labelpad=15)

plt.ylabel("Number of Job Listings", 
            fontdict={"color":"#333333", "fontweight":"bold", "fontsize":13}, 
            labelpad=15)

# Ticks
plt.xticks(rotation=25, ha='right', fontsize=12)
plt.yticks(fontsize=12)


# Value labels
for i, v in enumerate(y):
    plt.text(i, v + max(y)*0.008, str(v), ha='center', va='bottom', fontsize=12)

# Axis limits & spacing
plt.ylim(0, max(y) * 1.1)
plt.margins(y=0.05)

# Grid
plt.grid(axis='y', linestyle='--', alpha=0.2)

# Clean spines
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

# Show
plt.show()

KEY INSIGHTS OF EDA SECTION 01:

- Data Science domain has the highest number of job listings (High Demand), followed by Data Engineering and Data Analysis.
- This shows that Data Science Domain dominate the job market in this dataset compared to other data domains.

EDA SECTION 02: Identified Top Job Roles Within Each Data Domain Based on Job Listings

- This section identifies the top job roles within each data domain based on job listings
- The goal is to understand which roles are most in demand in each domain
- This helps identify the most common career paths across different data fields

In [ ]:
# Step 1: Compute job role frequency within each domain
# This gives us how many listings exist for each (job_category, job_title) pair

job_role_count = clean_dataset.groupby(["job_category", "job_title"]).size().reset_index(name="count")

In [ ]:

# Step02: Function to Plot Top Job Roles for Any Domain

def plot_domain(domain):

    # Filter data for selected domain and get top 5 job roles
    temp = job_role_count[job_role_count["job_category"] == domain].sort_values("count", ascending=False).head()

    # Wrap long job titles for better readability
    import textwrap
    wrapped_labels = [textwrap.fill(label, 12) for label in temp["job_title"]]

    # Create figure
    plt.figure(figsize=(12, 6), dpi=150)

    # Bar plot
    plt.bar(wrapped_labels, temp["count"], width=0.5, edgecolor="black", color="#52799C")

    # Titles and labels
    plt.title(
        f"Top {domain} Job Roles by Job Listings",
        color="#333333",
        fontweight="bold",
        fontsize=15,
        pad=10
    )
    plt.xlabel("Job Role", fontweight="bold", fontsize=13, labelpad=10, color="#333333")
    plt.ylabel("Number of Job Listings", fontweight="bold", fontsize=13, labelpad=10, color="#333333")

    # Axis formatting
    plt.xticks(rotation=0, fontsize=12)
    plt.yticks(fontsize=12)

    # Add value labels on bars
    for i, v in enumerate(temp["count"]):
        plt.text(
            i,
            v + max(temp["count"]) * 0.008,
            str(v),
            ha='center',
            va='bottom',
            fontsize=12
        )

    # Styling improvements
    plt.ylim(0, max(temp["count"]) * 1.1)
    plt.margins(y=0.05)
    plt.grid(axis='y', linestyle='--', alpha=0.3)

    # Remove unnecessary borders
    plt.gca().spines['top'].set_visible(False)
    plt.gca().spines['right'].set_visible(False)

    
    plt.show()

In [ ]:
plot_domain("Data Science")
plot_domain("Data Engineering")
plot_domain("Data Analysis")
plot_domain("Machine Learning & AI")
plot_domain("Business Intelligence")
plot_domain("Data Architecture")
plot_domain("Leadership & Management")
plot_domain("Other")

KEY INSIGHTS OF EDA SECTION 02:

- Each domain is dominated by one primary role (Data Scientist, Data Engineer, Data Analyst, ML Engineer, BI
Analyst, Data Architect, Research Engineer)
- Sharp drop after top role → highly concentrated demand
- Data Analyst shows highest entry-level demand
- ML roles are more niche
- Market favors standard roles over specialized ones.

EDA SECTION 03: Salary Comparison Across Data Domains

- This section compares salary across different data domains
- The goal is to see which domains pay more and how salaries differ
- This helps us understand where higher-paying opportunities exist in the data field

In [ ]:
#step01: Calculated median salary (USD) by job category and sorted to compare highest paying data domains
salary_by_domain = clean_dataset.groupby(clean_dataset["job_category"])["salary_in_usd"].median().reset_index(name="median_salary_usd")
salary_by_domain = salary_by_domain.sort_values("median_salary_usd", ascending=False)
salary_by_domain

In [ ]:
#step02: Visualized median salary (USD) across data domains for comparison

# Data
x = salary_by_domain["job_category"]
y = salary_by_domain["median_salary_usd"]

# Wrap long labels for better readability
import textwrap
wrapped_labels = [textwrap.fill(label, 12) for label in x]

# Create positions to increase spacing between bars
pos = np.arange(len(x)) * 1.3   # adjust (1.3–1.5) for more spacing

# Create figure
plt.figure(figsize=(16, 8), dpi=150)

# Bar plot
plt.bar(pos, y, width=0.7, edgecolor="black", color="#52799C")

# Titles and labels
plt.title(
    "Median Salary (USD) Across Data Domains",
    color="#333333",
    fontweight="bold",
    fontsize=24,
    pad=10
)
plt.xlabel("Data Domain", fontweight="bold", fontsize=20, labelpad=16, color="#333333")
plt.ylabel("Median Salary (USD)", fontweight="bold", fontsize=20, labelpad=16, color="#333333")

# Axis formatting
plt.xticks(pos, wrapped_labels, fontsize=17)
plt.yticks(fontsize=17)

# Add value labels (formatted as K)
for i, v in enumerate(y):
    label = f"{v/1000:.0f}K"   # 400000 → 400K
    plt.text(
        pos[i],
        v + max(y) * 0.008,
        label,
        ha='center',
        va='bottom',
        fontsize=17
    )

# Styling improvements
plt.ylim(0, max(y) * 1.1)
plt.margins(y=0.05)
plt.grid(axis='y', linestyle='--', alpha=0.3)

# Remove unnecessary borders
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

# Show plot
plt.tight_layout()
plt.show()

KEY INSIGHTS OF EDA SECTION 03:

- Machine Learning & AI offers the highest median salary, making it the highest paying domain
- Leadership & Management and Data Architecture domains also offer high salaries, showing strong pay for senior and specialized roles
- Data Analysis and Business Intelligence domains have the lowest salaries despite high demand, showing that demand != high pay.

EDA SECTION 04: Top 10 highest paying job roles based on median salary

- This section identifies the top 10 highest paying job roles based on median salary
- The goal is to compare salary levels across different job roles
- This helps us understand which data-related job roles offer the highest median salaries

In [ ]:
#step01: calculated top 10 median salary across all job roles

top_10 = (
    clean_dataset.groupby("job_title")["salary_in_usd"].median()
    .sort_values(ascending=False)
    .head(10)
)

top_10

In [ ]:
#step02: Created horizontal bar chart to visualize top 10 highest paying job roles by median salary (USD)

plt.figure(figsize=(16, 14), dpi=150)

#plotted horizontal bar chart with black borders for better visibility
x = top_10.plot(kind="barh", edgecolor="black", color="#52799C")

#added chart title and axis labels
plt.title("Top 10 Highest Paying Job Roles by Median Salary",
          color="#333333", fontweight="bold", fontsize=25, pad=30)

plt.xlabel("Median Salary (USD)",
           fontweight="bold", fontsize=23, labelpad=30, color="#333333")

plt.ylabel("Job Role",
           fontweight="bold", fontsize=23, labelpad=14, color="#333333")

#increased x-axis and y-axis tick label sizes
plt.xticks(fontsize=18)
plt.yticks(fontsize=18)

#added salary labels on each bar in K format
x.bar_label(
    x.containers[0],
    labels=[f"{l/1000:.0f}K" for l in top_10.values],
    padding=3,
    fontsize=18
)

#increased x-axis limit to create space for bar labels
plt.xlim(0, top_10.max() * 1.15)

#removed unnecessary top and right chart borders
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

#added grid lines
plt.grid(axis='y', linestyle='--', alpha=0.3)

#inverted y-axis to display highest salary at the top
plt.gca().invert_yaxis()

#adjusted layout spacing for cleaner visualization
plt.tight_layout()

#displayed final chart
plt.show()

In [ ]:
# KEY INSIGHTS OF EDA SECTION 04:

# - Highest salaries are concentrated in senior leadership roles (Managers, Directors, Heads)
# - Specialized architecture roles (AI, Cloud, AWS, Data Architect) also pay very high salaries
# - Entry-level and general roles are not present in top 10, showing salary is driven by experience and specialization

EDA SECTION 05: Demand vs Salary Analysis Across Data Domains

- This section compares job listing demand with median salary across different data domains
- The goal is to identify which domains offer the best balance between demand and pay
- This helps us understand the relationship between market demand and salary trends in the data field

In [ ]:
#step01: grouped data by job category and calculated median salary and job listing count for each data domain (job_category)

domain_analysis = (clean_dataset.groupby("job_category")
        .agg
        (median_salary_usd=("salary_in_usd", "median"),
        job_listing_count=("salary_in_usd", "count")
        )
        .reset_index()
        .sort_values(by="median_salary_usd", ascending=False)
        )

domain_analysis.columns


In [ ]:
#step02: visualizing Demand vs. Salary relationship on basis of Data domains (job_category)
x = domain_analysis["job_listing_count"]
y = domain_analysis["median_salary_usd"]
z = domain_analysis["job_category"]

#created figure with defined size
plt.figure(figsize=(10,6), dpi=150)

#plotted scatter plot using seaborn
ax = sns.scatterplot(x=x, y=y, data=domain_analysis, s=200, color="#52799C")

#expanded x-axis range for better label spacing
plt.xlim(0, x.max() * 1.25)

#added title and axis labels
plt.title("Demand vs Salary Across Data Domains",
            color="#333333", fontweight="bold", fontsize=15, pad=13)

plt.xlabel("Number of Job Listings",
            color="#333333", fontweight="bold", fontsize=12)

plt.ylabel("Median Salary (USD)",
            color="#333333", fontweight="bold", fontsize=12)

#adjusted label spacing for axis titles
ax.xaxis.labelpad = 15
ax.yaxis.labelpad = 15

#added domain labels on each scatter point with custom positioning
for x_val, y_val, label in zip(x, y, z):
    if label == "Data Architecture":
        ax.text(x_val, y_val - 4000, label, fontsize=10)
    else:
        ax.text(x_val + 40, y_val, label, fontsize=10, alpha=0.9)

#removed top and right borders for cleaner look
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

#added light grid for better readability
plt.grid(axis="y", linestyle='--', alpha=0.2)

#displayed final visualization
plt.show()

KEY INSIGHTS OF EDA SECTION 05:

- Machine Learning & AI offers the highest salary despite moderate job demand, making it a high-value niche domain
- Data Science & Data Engineering have high demand but moderate salaries
- Data Analysis and Business Intelligence have high demand but the lowest salaries, showing demand does not always translate to high pay
- Leadership & Management and Data Architecture show low-to-moderate demand but strong salaries, highlighting premium pay for senior/specialized roles
- Overall, the best opportunities exist in domains that balance specialization (AI, Architecture) with experience-driven roles

EDA SECTION 06: Demand Trend Across Data Domains (2020–2024)

- This section analyzes how demand for different data domains changed between 2020 and 2024
- The goal is to identify which domains experienced growth, stability, or decline in job demand
- This helps us understand hiring trends and the changing popularity of data-related domains over time

In [ ]:
#step01: Calculated yearly job demand across data domains by counting job listings for each domain from 2020–2024

domain_demand_trend = clean_dataset.groupby(["work_year", "job_category"]).size().reset_index(name="job_listing_count")

# fix data type issue (remove .0 from years)
domain_demand_trend["work_year"] = domain_demand_trend["work_year"].astype(str)

domain_demand_trend.head(10)

In [ ]:
#step02: Visualized demand trends across data domains from 2020–2024 using a Line Chart

fig, ax = plt.subplots(figsize=(12,6), dpi=150)

sns.lineplot(
    data=domain_demand_trend,
    x="work_year",
    y="job_listing_count",
    hue="job_category",
    marker="o",
    markersize=8,
    linewidth=2,
    ax=ax
)

# Title
ax.set_title(
    "Demand Growth Across Data Domains (2020–2024)",
    color="#333333",
    fontweight="bold",
    fontsize=17,
    pad=17
)

# Axis labels with proper spacing
ax.set_xlabel(
    "Year",
    color="#333333",
    fontweight="bold",
    fontsize=14,
    labelpad=12
)

ax.set_ylabel(
    "Number of Job Listings",
    color="#333333",
    fontweight="bold",
    fontsize=14,
    labelpad=12
)

ax.tick_params(axis='both', labelsize=13)

# Grid styling (clean look)
ax.grid(True, linestyle="--", alpha=0.25)

# Tick spacing (better breathing space)
ax.tick_params(axis='x', pad=6)
ax.tick_params(axis='y', pad=6)

# Legend outside (clean + no overlap)
ax.legend(
    title="Data Domain",
    loc="upper left",
    frameon=True,
    title_fontsize=11,
    fontsize=10
)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.show()

KEY INSIGHTS OF EDA SECTION 06:

- Job demand across all data domains grew consistently from 2020 to 2023
- Data Science recorded the highest job demand throughout the period and experienced the sharpest growth by 2023
- Data Engineering and Data Analysis also experienced significant demand growth
- Machine Learning & AI demand increased rapidly after 2022, reflecting growing industry adoption of AI technologies
- Leadership & Management, Business Intelligence, and Data Architecture maintained comparatively lower but stable demand levels
- Most domains showed a noticeable drop in 2024, which may indicate incomplete yearly data or changing market conditions

EDA SECTION 07: Salary Trend Across Data Domains (2020–2024)

- This section analyzes how median salary across different data domains changed between 2020 and 2024
- The goal is to identify which domains experienced salary growth, stability, or decline over time
- This helps us understand compensation trends and how the market value of different domains evolved over the years

In [ ]:
#step01: Grouped data by year and job category, then calculates median salary for each group
domain_salary_trend = clean_dataset.groupby(["work_year", "job_category"])["salary_in_usd"].median().reset_index(name="median_salary")

# Converted year to string for better visualization in plots/charts
domain_salary_trend["work_year"] = domain_salary_trend["work_year"].astype(str)

domain_salary_trend.head(10)

In [ ]:
#step02: Visualized salary trend across data domains from 2020-2024 using a Line Chart

fig, ax = plt.subplots(figsize=(12,8), dpi=150)

sns.lineplot(
    data=domain_salary_trend,
    x="work_year",
    y="median_salary",
    hue="job_category",
    marker="o",
    markersize=8,
    linewidth=2,
    ax=ax
)

# Title
ax.set_title(
    "Median Salary Trends Across Data Domains (2020–2024)",
    color="#333333",
    fontweight="bold",
    fontsize=20,
    pad=17
)

# Axis labels with proper spacing
ax.set_xlabel(
    "Year",
    color="#333333",
    fontweight="bold",
    fontsize=17,
    labelpad=16
)

ax.set_ylabel(
    "Median Salary (USD)",
    color="#333333",
    fontweight="bold",
    fontsize=17,
    labelpad=16
)

ax.tick_params(axis='both', labelsize=16)

# Grid styling (clean look)
ax.grid(True, linestyle="--", alpha=0.25)

# Tick spacing (better breathing space)
ax.tick_params(axis='x', pad=6)
ax.tick_params(axis='y', pad=6)

# Legend outside (clean + no overlap)
ax.legend(
    bbox_to_anchor=(1.05, 1),
    title="Data Domain",
    loc="upper left",
    frameon=True,
    title_fontsize=13,
    fontsize=12
)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.show()

KEY INSIGHTS OF EDA SECTION 07

- ML & AI shows strongest growth (~100K → ~185K), highest pay for AI skills
- Data Science grows till 2023, slight dip in 2024 (possible saturation)
- Data Engineering shows steady growth, stabilizing ~137–140K
- Data Analysis remains flat (~78K → ~110K), slow salary growth
- Leadership & Management peaks in 2024 (~180K), high-value roles
- Data Architecture slightly declines after peak, still stable niche role
- BI grows till 2023, declines in 2024 (tool shift impact)
- Overall: AI/ML leads salary growth, traditional roles are plateauing

COMMON RELATIONSHIP INSIGHTS – EDA SECTION 06 & 07

- ML & AI shows strongest growth in both demand and salary, leading overall
- Data Engineering shows steady growth in both, indicating stable demand and salary trend
- Data Science has high demand but slightly weaker salary growth in 2024, suggesting early saturation
- Data Analysis shows moderate demand but low salary growth, reflecting entry-level dominance
- Leadership & Management shows strong salary performance with stable demand, indicating high-value strategic roles
- BI and Data Architecture show stable to declining trends in both demand and salary, indicating mature/shifting domains
- Overall: strong positive link between demand and salary, with AI/ML leading and traditional roles lagging

EDA SECTION 08: Job Demand Across Experience Levels

- This section analyzes how job demand varies across different experience levels
- The goal is to identify which experience level companies hire most
- This helps understand hiring preferences of companies and which career stages have the highest opportunities

In [ ]:
#step01: Calculated job listing count by Experience Level

job_demand_by_experience = (
    clean_dataset.groupby("experience_level")
    .size().reset_index(name="job_listing_count")
    .sort_values("job_listing_count", ascending=False)
)

job_demand_by_experience

In [ ]:
#step02: Visualized job demand across experience levels

# Data
x = job_demand_by_experience["experience_level"]
y = job_demand_by_experience["job_listing_count"]

# Wrap long labels for better readability
import textwrap
wrapped_labels = [textwrap.fill(label, 12) for label in x]

# Create positions to increase spacing between bars
pos = np.arange(len(x)) * 1.3   # adjust (1.3–1.5) for more spacing

# Create figure
plt.figure(figsize=(16, 8), dpi=150)

# Bar plot
plt.bar(pos, y, width=0.7, edgecolor="black", color="#52799C")

# Titles and labels
plt.title(
    "Job Demand Across Experience Levels",
    color="#333333",
    fontweight="bold",
    fontsize=22,
    pad=10
)
plt.xlabel("Experience Level", fontweight="bold", fontsize=18, labelpad=14, color="#333333")
plt.ylabel("Number of Job Listings", fontweight="bold", fontsize=18, labelpad=20, color="#333333")

# Axis formatting
plt.xticks(pos, wrapped_labels, fontsize=17)
plt.yticks(fontsize=17)

# Add value labels
plt.bar_label(plt.gca().containers[0],
            fontsize=17,        
            padding=3) 

# Styling improvements
plt.ylim(0, max(y) * 1.1)
plt.margins(y=0.05)
plt.grid(axis='y', linestyle='--', alpha=0.3)

# Remove unnecessary borders
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

# Show plot
plt.show()

KEY INSIGHTS OF EDA SECTION 08:

- Senior-level roles dominate job demand (4103), showing strong preference for experienced talent
- Mid-level roles follow (1668), indicating steady demand for skilled professionals
- Entry-level (565) and Executive-level (254) roles have much lower demand, showing limited openings at both early and top leadership stages

EDA SECTION 09: Salary Distribution Across Experience Levels

- This section analyzes how salaries vary across different experience levels
- The goal is to identify which experience levels receive the highest and lowest compensation
- This helps understand salary progression across career stages in the data job market

In [ ]:
#step01: Calculated Median Salary by Experience Level

median_salary_by_experience = (
    clean_dataset.groupby("experience_level")["salary_in_usd"]
    .median().reset_index(name="median_salary")
    .sort_values("median_salary", ascending=False)
)
median_salary_by_experience

In [ ]:
#step02: Visualized median salary (USD) Across Experience Levels

# Data
x = median_salary_by_experience["experience_level"]
y = median_salary_by_experience["median_salary"]

# Wrap long labels for better readability
import textwrap
wrapped_labels = [textwrap.fill(label, 12) for label in x]

# Create positions to increase spacing between bars
pos = np.arange(len(x)) * 1.3  

# Create figure
plt.figure(figsize=(16, 8), dpi=150)

# Bar plot
plt.bar(pos, y, width=0.7, edgecolor="black", color="#52799C")

# Titles and labels
plt.title(
    "Median Salary Across Experience Levels",
    color="#333333",
    fontweight="bold",
    fontsize=22,
    pad=10
)
plt.xlabel("Experience Level", fontweight="bold", fontsize=18, labelpad=10, color="#333333")
plt.ylabel("Median Salary (USD)", fontweight="bold", fontsize=18, labelpad=14, color="#333333")

# Axis formatting
plt.xticks(pos, wrapped_labels, fontsize=17)
plt.yticks(fontsize=17)

# Add value labels (formatted as K)
for i, v in enumerate(y):
    label = f"{v/1000:.0f}K"  
    plt.text(
        pos[i],
        v + max(y) * 0.008,
        label,
        ha='center',
        va='bottom',
        fontsize=17
    )

# Styling improvements
plt.ylim(0, max(y) * 1.1)
plt.margins(y=0.05)
plt.grid(axis='y', linestyle='--', alpha=0.3)

# Remove unnecessary borders
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

# Show plot
plt.show()

KEY INSIGHTS OF EDA SECTION 09:

- Executive-level roles have the highest median salary (~184K), reflecting premium pay for leadership positions.
- Senior-level professionals earn significantly higher salaries (~154K) than mid and entry-level roles.
- Mid-level salaries (~106K) show strong growth from entry-level, highlighting career progression benefits.
- Entry-level roles have the lowest median salary (~75K), representing the starting stage of data careers.

RELATIONSHIP INSIGHTS – EDA SECTION 08 & 09

- Senior-level roles show the highest job demand along with strong salary levels, making them the most balanced career stage.
- Executive-level roles have the highest salaries but the lowest demand, reflecting fewer but highly paid leadership positions.
- Mid-level roles maintain steady demand and salary growth, indicating strong career progression opportunities.
- Entry-level roles have lower demand and the lowest salaries, showing limited openings for freshers in the market.
- Overall: higher experience levels are associated with higher salaries, but not always with higher job demand.

EDA SECTION 10: Work Model Preferences Across Data Domains

- This section analyzes how work models vary across different data domains
- The goal is to identify which domains prefer remote, hybrid, or on-site work
- This helps understand workplace trends and flexibility across data roles

In [ ]:
# STEP 01: Calculated Work Mode Preferences Across Data Domains

work_mode_preferences = (
    clean_dataset.groupby(["job_category", "work_models"])
    .size()
    .reset_index(name="work_mode_count")
)

# sort within each domain by work_model_count (desc)
sorted_df = work_mode_preferences.sort_values(
    ["job_category", "work_mode_count"],
    ascending=[True, False]
)

sorted_df

In [ ]:
#step02: visualized how different data domains prefer work modes (On-site, Remote, Hybrid)

# Set figure size and resolution for better clarity
plt.figure(figsize=(12, 6), dpi=150)

# Create bar plot to compare job categories vs work model counts
ax = sns.barplot(
    data=sorted_df,
    x="job_category", 
    y="work_mode_count", 
    hue="work_models",              # separates bars by work type (Remote/On-site/Hybrid)
    palette="Blues_d",              # blue color theme for clean visual look
    width=0.8,                      # controls bar thickness
    edgecolor="black",              # adds border to bars
    linewidth=0.7                   # border thickness
)

# Add chart title
plt.title(
    "Work Mode Preferences Across Data Domains",
    fontsize=16,
    fontweight="bold",
    color="#333333",
    pad=10
)

# X-axis label
plt.xlabel(
    "Data Domain",
    fontsize=13,
    fontweight="bold",
    color="#333333",
    labelpad=10
)

# Y-axis label
plt.ylabel(
    "Number of Job Postings",
    fontsize=13,
    fontweight="bold",
    color="#333333",
    labelpad=10
)

# Wrap long x-axis labels to improve readability
import textwrap
ax.set_xticks(ax.get_xticks())
ax.set_xticklabels(
    [textwrap.fill(label.get_text(), 12) for label in ax.get_xticklabels()],
    rotation=0
)

# Adjust tick label sizes
ax.tick_params(axis='x', labelsize=11)
ax.tick_params(axis='y', labelsize=11)

# Add legend for work model categories
ax.legend(title="Work Modes")

# Add light grid lines for better readability of values
plt.grid(axis='y', linestyle='--', alpha=0.3)

# Remove unnecessary plot borders (top and right)
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

# Display final plot
plt.show()

KEY INSIGHTS OF EDA SECTION 10:

- On-site work mode dominates across all data domains, especially in Data Science and Data Engineering
- Remote work mode is the second most common option, showing strong demand for flexible roles in tech fields
- Hybrid work mode is minimal across all domains, indicating it is still not widely adopted in data-related jobs

EDA SECTION 11: Median Salary by Work Mode

- This section analyzes salary differences across Remote, Hybrid, and On-site jobs
- The goal is to check which work model offers higher median salaries
- This helps understand whether flexible work options are linked to better pay in data jobs

In [ ]:
#step01: Calculated median salary for each work model

median_salary_workmode = (clean_dataset.groupby("work_models")
                            ["salary_in_usd"].median()
                            .reset_index(name="median_salary")
                            .sort_values("median_salary", ascending=False))
median_salary_workmode

In [ ]:
#step02: Visualized median salary across different work models using a bar chart for salary comparison

# Data
x = median_salary_workmode["work_models"]
y = median_salary_workmode["median_salary"]

# Create figure
plt.figure(figsize=(10, 5), dpi=150)

# Bar plot
plt.bar(x, y, width=0.5, edgecolor="black", color="#52799C")

# Titles and labels
plt.title(
    "Median Salary Across Work Modes",
    color="#333333",
    fontweight="bold",
    fontsize=15,
    pad=15
)
plt.xlabel("Work Mode", fontweight="bold", fontsize=12, labelpad=10, color="#333333")
plt.ylabel("Median Salary (USD)", fontweight="bold", fontsize=12, labelpad=14, color="#333333")

# Axis formatting
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

# Add value labels (formatted as K)
for i, v in enumerate(y):
    label = f"{v/1000:.0f}K"  
    plt.text(
        i,
        v + max(y) * 0.008,
        label,
        ha='center',
        va='bottom',
        fontsize=12
    )

# Styling improvements
plt.ylim(0, max(y) * 1.1)
plt.margins(y=0.05)
plt.grid(axis='y', linestyle='--', alpha=0.3)

# Remove unnecessary borders
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
# Show plot
plt.show()

KEY INSIGHTS OF EDA SECTION 11:

- Even though On-site work dominates across all data domains (EDA 10), it also offers the highest median salary (EDA 11), showing strong consistency between demand and pay
- Remote work is the second most common work mode and also provides nearly similar salaries to On-site roles, making it a strong flexible and high-paying option
- Hybrid work is both the least adopted (EDA 10) and lowest paying (EDA 11), indicating it is the weakest work mode in both demand and compensation

EDA SECTION 12: Top Hiring Countries / Company Locations

- This section analyzes which countries or locations have the highest number of job opportunities
- The goal is to identify key regions where most hiring activity is happening in data roles
- This helps understand global job demand and major hiring hubs in the data industry

In [ ]:
#step01: Calculated top 10 company locations with highest median salaries

median_salary_bylocation = (clean_dataset
                            .groupby("company_location")
                            ["salary_in_usd"].median()
                            .reset_index(name="median_salary")
                            .sort_values("median_salary", ascending=False)
                            .head(10))

median_salary_bylocation

In [ ]:
#step02: Visualized top-paying company locations by Median salary

# Data
x = median_salary_bylocation["company_location"]
y = median_salary_bylocation["median_salary"]

# Wrap long labels for better readability
import textwrap
wrapped_labels = ["\n".join(label.split()) for label in x]

# Create positions to increase spacing between bars
pos = np.arange(len(x)) * 2.0   # adjust (1.3–1.5) for more spacing

# Create figure
plt.figure(figsize=(16, 8), dpi=150)

# Bar plot
plt.bar(pos, y, width=0.9, edgecolor="black", color="#52799C")

# Titles and labels
plt.title(
    "Top 10 Company Locations by Median Salary",
    color="#333333",
    fontweight="bold",
    fontsize=26,
    pad=10
)
plt.xlabel("Company Location", fontweight="bold", fontsize=20, labelpad=25, color="#333333")
plt.ylabel("Median Salary (USD)", fontweight="bold", fontsize=20, labelpad=25, color="#333333")

# Axis formatting
plt.xticks(pos, wrapped_labels, fontsize=17)
plt.yticks(fontsize=17)

# Add value labels (formatted as K)
for i, v in enumerate(y):
    label = f"{v/1000:.0f}K"   # 400000 → 400K
    plt.text(
        pos[i],
        v + max(y) * 0.008,
        label,
        ha='center',
        va='bottom',
        fontsize=17
    )

# Styling improvements
plt.ylim(0, max(y) * 1.1)
plt.margins(y=0.05)
plt.grid(axis='y', linestyle='--', alpha=0.3)

# Remove unnecessary borders
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

# Show plot
plt.tight_layout()
plt.show()

KEY INSIGHTS OF EDA SECTION 12:

- Qatar has the highest median salary among all locations, standing out as the top-paying country
- Puerto Rico, United States, Saudi Arabia, and Canada also offer high-paying data job opportunities
- High salaries are spread across different countries, showing that top-paying roles are not limited to one region

EDA SECTION 13: Company Size Hiring Distribution Across Data Domains

- This section analyzes how job opportunities are distributed across company sizes (Small, Medium, Large) within different data domains
- The goal is to see which company size hires the most in each domain
- This helps understand hiring patterns based on company scale in data roles

In [ ]:
#step01: Calculate job count by company size across each job category

company_size_hiring_count = (clean_dataset
                            .groupby(["job_category", "company_size"])
                            .size()
                            .reset_index(name="job_count")
                            .sort_values(["job_category", "job_count"], ascending=[True, False]))

company_size_hiring_count

In [ ]:
#step02: Visualized how company sizes (Small, Medium, Large) hire across data domains

# Set figure size and resolution for better clarity
plt.figure(figsize=(12, 6), dpi=150)

# Create bar plot to compare job categories vs work model counts
ax = sns.barplot(
    data=company_size_hiring_count,
    x="job_category", 
    y="job_count", 
    hue="company_size",              # separates bars by work type (Remote/On-site/Hybrid)
    palette="Blues_d",              # blue color theme for clean visual look
    width=0.8,                      # controls bar thickness
    edgecolor="black",              # adds border to bars
    linewidth=0.7                   # border thickness
)

# Add chart title
plt.title(
    "Company Size Hiring Across Data Domains",
    fontsize=16,
    fontweight="bold",
    color="#333333",
    pad=10
)

# X-axis label
plt.xlabel(
    "Data Domain",
    fontsize=13,
    fontweight="bold",
    color="#333333",
    labelpad=10
)

# Y-axis label
plt.ylabel(
    "Number of Job Postings",
    fontsize=13,
    fontweight="bold",
    color="#333333",
    labelpad=10
)

# Wrap long x-axis labels to improve readability
import textwrap
ax.set_xticks(ax.get_xticks())
ax.set_xticklabels(
    [textwrap.fill(label.get_text(), 12) for label in ax.get_xticklabels()],
    rotation=0
)

# Adjust tick label sizes
ax.tick_params(axis='x', labelsize=11)
ax.tick_params(axis='y', labelsize=11)

# Add legend for work model categories
ax.legend(title="Work Modes")

# Add light grid lines for better readability of values
plt.grid(axis='y', linestyle='--', alpha=0.3)

# Remove unnecessary plot borders (top and right)
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

# Display final plot
plt.show()

KEY INSIGHTS OF EDA SECTION 13:

- Medium-sized companies hire the most across all data domains
- Large companies hire second most, but their numbers are much lower than medium companies
- Small companies have the least hiring in all domains, showing limited job opportunities from smaller firms

EDA SECTION 14: Median Salary by Company Size

- This section analyzes how median salaries vary across different company sizes (Small, Medium, Large)
- The goal is to compare salary trends between startups, medium-sized companies, and large companies
- This helps understand how company size influences pay in data-related roles

In [ ]:
#step01: Calculated median salary across different company sizes

median_salary_company_size = (clean_dataset
                            .groupby("company_size")
                            ["salary_in_usd"].median()
                            .reset_index(name="median_salary")
                            .sort_values("median_salary", ascending=False))

median_salary_company_size

In [ ]:
#step02: Visualized median salary across different company sizes using a bar chart

# Data
x = median_salary_company_size["company_size"]
y = median_salary_company_size["median_salary"]

# Create figure
plt.figure(figsize=(10, 5), dpi=150)

# Bar plot
plt.bar(x, y, width=0.5, edgecolor="black", color="#52799C")

# Titles and labels
plt.title(
    "Median Salary Across Company Sizes",
    color="#333333",
    fontweight="bold",
    fontsize=15,
    pad=15
)
plt.xlabel("Company Size", fontweight="bold", fontsize=12, labelpad=10, color="#333333")
plt.ylabel("Median Salary (USD)", fontweight="bold", fontsize=12, labelpad=14, color="#333333")

# Axis formatting
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

# Add value labels (formatted as K)
for i, v in enumerate(y):
    label = f"{v/1000:.0f}K"  
    plt.text(
        i,
        v + max(y) * 0.008,
        label,
        ha='center',
        va='bottom',
        fontsize=12
    )

# Styling improvements
plt.ylim(0, max(y) * 1.1)
plt.margins(y=0.05)
plt.grid(axis='y', linestyle='--', alpha=0.3)

# Remove unnecessary borders
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
# Show plot
plt.show()

KEY INSIGHTS OF EDA SECTION 14:

- Medium-sized companies not only hire the most (EDA 13) but also offer the highest median salary, making them the strongest overall option
- Large companies provide the second highest median salary, but still pay less than medium-sized companies
- Small companies have both the lowest hiring and lowest median salary, showing limited opportunities and lower pay

EDA SECTION 15: Employment Type Distribution (Full-time / Contract / Part-time)

- This section analyzes how jobs are split across Full-time, Contract, and Part-time roles
- The goal is to identify which employment type is most common in the data job market
- This helps understand overall hiring preference for different job types in the industry

In [ ]:
#step01: Calculated job count for each employment type

employment_distribution = (
                clean_dataset
                .groupby("employment_type")
                .size()
                .reset_index(name="job_count")
                .sort_values("job_count", ascending=False))

employment_distribution

In [ ]:
#step02: Visualized employment type distribution to identify the most preferred job type in the data job market

# Data
x = employment_distribution["employment_type"]
y = employment_distribution["job_count"]

# Wrap long labels for better readability
import textwrap
wrapped_labels = [textwrap.fill(label, 12) for label in x]

# Create positions to increase spacing between bars
pos = np.arange(len(x)) * 1.3   # adjust (1.3–1.5) for more spacing

# Create figure
plt.figure(figsize=(16, 8), dpi=150)

# Bar plot
plt.bar(pos, y, width=0.7, edgecolor="black", color="#52799C")

# Titles and labels
plt.title(
    "Employment Type Distribution in Data Jobs",
    color="#333333",
    fontweight="bold",
    fontsize=22,
    pad=10
)
plt.xlabel("Employment Type", fontweight="bold", fontsize=18, labelpad=25, color="#333333")
plt.ylabel("Number of Job postings", fontweight="bold", fontsize=18, labelpad=25, color="#333333")

# Axis formatting
plt.xticks(pos, wrapped_labels, fontsize=17)
plt.yticks(fontsize=17)

# Add value labels
plt.bar_label(plt.gca().containers[0],
            fontsize=17,        
            padding=3) 

# Styling improvements
plt.ylim(0, max(y) * 1.1)
plt.margins(y=0.05)
plt.grid(axis='y', linestyle='--', alpha=0.3)

# Remove unnecessary borders
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

# Show plot
plt.show()

KEY INSIGHTS OF EDA SECTION 15:

- Full-time jobs dominate the data job market, showing very high demand for permanent roles
- Contract, Part-time, and Freelance roles are very limited compared to Full-time jobs
- The data industry strongly prefers stable long-term employment over flexible work types

EDA SECTION 16: Final Insights & Summary

- This section summarizes the most important findings from all previous EDA sections
- The goal is to highlight major trends in job demand, salaries, work models, company size, and hiring patterns
- This helps provide a clear overall understanding of data job market and industry trends from 2020 to 2024

1.  Data Science emerged as the most dominant domain in the dataset,
    with the highest job demand from 2020–2024. Data Engineering and Data Analysis
    also showed consistently strong demand, making them key hiring areas in the data industry.

2.  Machine Learning & AI stood out as the highest-paying and fastest-growing domain.
    Although its job demand was lower than Data Science, its salary growth increased rapidly,
    highlighting strong market value for AI-related skills.

3.  Data Analysis and Business Intelligence had high hiring demand but comparatively lower salaries,
    showing that high demand does not always translate into high pay, especially in more common or entry-focused roles.

4.  Leadership, Management, and specialized Architecture roles offered very high salaries
    despite lower hiring demand, indicating that senior expertise and niche skills are highly valued.

5.  Overall job demand across most domains increased from 2020–2023, reflecting strong industry growth,
    but several domains showed a decline in 2024, which may indicate incomplete data or market shifts.

6.  Experience level had a strong impact on both salary and hiring.
    Senior roles had the highest demand, while Executive roles earned the highest salaries.
    Entry-level roles had lower salaries and fewer opportunities.

7.  On-site work was the most common work model across all domains.
    Remote jobs were the second most common and also well-paid,
    while Hybrid roles remained limited in both usage and salary.

8.  Medium-sized companies provided the strongest overall opportunities,
    with the highest hiring and median salaries.
    Small companies showed the lowest hiring and salary levels.

9.  Full-time employment dominated the job market,
    while Contract, Part-time, and Freelance roles remained limited,
    showing a strong preference for stable long-term employment.

10. High-paying data roles were distributed across multiple countries including
    the United States, Canada, Qatar, Saudi Arabia, and others,
    showing global demand for data professionals.

11. Overall, the data industry showed strong growth from 2020–2024,
    especially in AI, Data Science, and Engineering domains,
    with a clear preference for experienced professionals, specialized skills, and full-time roles.